# MCP Unstructured v7.5\n\nLean install without unstructured-inference, plus an optional hosted VLM toggle.\n

In [4]:
!git clone https://github.com/dshipley71/mcp-servers.git

Cloning into 'mcp-servers'...
remote: Enumerating objects: 475, done.
remote: Counting objects: 100% (246/246), done.
remote: Compressing objects: 100% (189/189), done.
remote: Total 475 (delta 124), reused 66 (delta 33), pack-reused 229 (from 1)
Receiving objects: 100% (475/475), 296.21 KiB | 8.00 MiB/s, done.
Resolving deltas: 100% (231/231), done.


In [1]:
%cd /content/mcp-servers/mcp-unstructured

/content/mcp-servers/mcp-unstructured


In [9]:
%cat requirements.txt

numpy==1.26.4
unstructured
pdfminer.six
requests
python-docx
python-pptx
openpyxl
pypdf
pdf2image
pytesseract
unstructured-pytesseract


In [10]:
!apt-get update -y
!apt-get install -y poppler-utils tesseract-ocr libmagic-dev
!pip install --no-cache-dir -r requirements.txt
print('IMPORTANT: Restart the runtime now, then continue with the remaining cells.')


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 1s (3,304 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1

IMPORTANT: Restart the runtime now, then continue with the remaining cells.


In [4]:
import os, sys
os.environ['ALLOWED_ROOT'] = '/content'
print('Working directory:', os.getcwd())
print('ALLOWED_ROOT =', os.environ['ALLOWED_ROOT'])


Working directory: /content
ALLOWED_ROOT = /content


In [5]:
USE_VLM = False
VLM_PROVIDER = 'openai'
VLM_MODEL = 'gpt-4o'
# Optional: uncomment and set these only if you want hosted VLM mode
# os.environ['UNSTRUCTURED_API_URL'] = 'https://<your-endpoint>'
# os.environ['UNSTRUCTURED_API_KEY'] = '<your-api-key>'
print({'USE_VLM': USE_VLM, 'VLM_PROVIDER': VLM_PROVIDER, 'VLM_MODEL': VLM_MODEL})

{'USE_VLM': False, 'VLM_PROVIDER': 'openai', 'VLM_MODEL': 'gpt-4o'}


In [17]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))


Saving AGENTS_updated.md to AGENTS_updated.md
['AGENTS_updated.md']


In [7]:
%pwd

'/content'

In [8]:
%cd /content/mcp-servers/mcp-unstructured/src

/content/mcp-servers/mcp-unstructured/src


In [9]:
import threading
from mcp_unstructured import server
thread = threading.Thread(target=lambda: server.run_http(8000), daemon=True)
thread.start()
print('Server started on http://localhost:8000')


Server running on port 8000
Server started on http://localhost:8000


In [10]:
import requests
resp = requests.post('http://localhost:8000', json={'tool': 'health'}, timeout=30)
print(resp.status_code)
print(resp.json())


200
{'result': {'status': 'ok', 'numpy_version': '1.26.4', 'inference_enabled': False, 'vlm_available_via_api': False, 'default_vlm_provider': 'openai', 'default_vlm_model': 'gpt-4o'}}


In [25]:
sample_path = '/content/mcp-servers/mcp-unstructured/src/' + list(uploaded.keys())[0]
print('Sample path:', sample_path)


Sample path: /content/mcp-servers/mcp-unstructured/src/AGENTS_updated.md


In [23]:
%ls /content/mcp-servers/mcp-unstructured/src

AGENTS_updated.md  __init__.py  mcp_unstructured/


In [15]:
!pip install pi_heif

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.4 MB/s eta 0:00:00


In [37]:
import requests
import rich
from rich.pretty import pprint
payload = {
  'tool': 'parse_file',
  'path': '/content/mcp-servers/mcp-unstructured/src/AGENTS_updated.md', #sample_path,
  'route': 'fast',
  'chunking_strategy': 'basic',
  'vlm_mode': USE_VLM,
  'vlm_model_provider': VLM_PROVIDER,
  'vlm_model': VLM_MODEL,
}

resp = requests.post('http://localhost:8000', json=payload, timeout=600 if USE_VLM else 180)


In [40]:
from rich.panel import Panel
from rich.console import Console

console = Console()
result = resp.json()["result"]

console.print(Panel(result["text"][:1000], title="Document Text"))

for i, chunk in enumerate(result["chunks"][:]):
    console.print(Panel(chunk["text"], title=f"Chunk {i}"))

print(f"Total Chunks {len(result["chunks"])}")

╭───────────────────────────────────────────────── Document Text ─────────────────────────────────────────────────╮
│ AGENTS.md                                                                                                       │
│                                                                                                                 │
│ Purpose                                                                                                         │
│                                                                                                                 │
│ This project is an evidence-constrained intelligence platform that retrieves recent signals from multiple       │
│ sources, ranks and deduplicates them, generates grounded synthesis, creates derivative analyst outputs, exports │
│ structured artifacts, and exposes the workflow through a FastAPI backend and Gradio frontend.                   │
│                                                                                                                 │
│ This file is synchronized to the current codebase as implemented in the notebook-generated modules.             │
│                                                                                                                 │
│ Current Code                                                                                                    │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│ Aligned Agent Topology                                                                                          │
│                                                                                                                 │
│ The code currently implements the following agents under:                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/retrieval.py                                                                    │
│                                                                                                                 │
│ backend/src/intelligence/agents/ranking.py                                                                      │
│                                                                                                                 │
│ backend/src/intelligence/agents/synthesis.py                                                                    │
│                                                                                                                 │
│ backend/src/intelligence/agents/content.py                                                                      │
│                                                                                                                 │
│ backend/src/intelligence/agents/export.py                                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/review.py                                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/orchestrator.py                                                                 │
│                                                                                                                 │
│ Important synchronization note                                                                                  │
│                                                       

╭──────────────────────────────────────────────────── Chunk 0 ────────────────────────────────────────────────────╮
│ AGENTS.md                                                                                                       │
│                                                                                                                 │
│ Purpose                                                                                                         │
│                                                                                                                 │
│ This project is an evidence-constrained intelligence platform that retrieves recent signals from multiple       │
│ sources, ranks and deduplicates them, generates grounded synthesis, creates derivative analyst outputs, exports │
│ structured artifacts, and exposes the workflow through a FastAPI backend and Gradio frontend.                   │
│                                                                                                                 │
│ This file is synchronized to the current codebase as implemented in the notebook-generated modules.             │
│                                                                                                                 │
│ Current Code                                                                                                    │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│ Aligned Agent Topology                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Chunk 1 ────────────────────────────────────────────────────╮
│ The code currently implements the following agents under:                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/retrieval.py                                                                    │
│                                                                                                                 │
│ backend/src/intelligence/agents/ranking.py                                                                      │
│                                                                                                                 │
│ backend/src/intelligence/agents/synthesis.py                                                                    │
│                                                                                                                 │
│ backend/src/intelligence/agents/content.py                                                                      │
│                                                                                                                 │
│ backend/src/intelligence/agents/export.py                                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/review.py                                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/orchestrator.py                                                                 │
│                                                                                                                 │
│ Important synchronization note                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Chunk 2 ────────────────────────────────────────────────────╮
│ The earlier specification included IntakeAgent and SourcePlanningAgent. Those are not currently implemented as  │
│ standalone agents in code.                                                                                      │
│                                                                                                                 │
│ Their responsibilities are currently handled inside orchestrator.py by: - constructing ResearchRequest -        │
│ assigning defaults for enabled sources - passing request parameters directly into retrieval                     │
│                                                                                                                 │
│ To stay synchronized with the codebase, this AGENTS file documents the agents that actually exist today.        │
│                                                                                                                 │
│ Agent Contracts                                                                                                 │
│                                                                                                                 │
│ 1. RetrievalAgent                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Chunk 3 ────────────────────────────────────────────────────╮
│ Module: backend/src/intelligence/agents/retrieval.py                                                            │
│                                                                                                                 │
│ Purpose: Execute multi-source retrieval and persist raw evidence artifacts.                                     │
│                                                                                                                 │
│ Inputs:                                                                                                         │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│  run_dir: str                                                                                                   │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│  query: str                                                                                                     │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│  days: int = 30                                                                                                 │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│  enabled_sources: list | None                                                                                   │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│  verbose: bool = False                                                                                          │
│                                                                                                                 │
│ Primary responsibilities: - call retrieval_service.retrieve(...) - execute enabled source adapters - collect    │
│ raw source items by source - persist raw retrieval output - persist retrieval summary counts                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────── Chunk 17 ────────────────────────────────────────────────────╮
│ A run is considered complete only when: - processed/ranked_items.json exists - synthesis/package.json exists -  │
│ outputs/storyboard.md exists - export/export.json exists - review/review_report.json exists                     │
│                                                                                                                 │
│ Synchronization Gaps Between Spec and Code                                                                      │
│                                                                                                                 │
│ The following earlier-spec items are not yet fully implemented as standalone code components:                   │
│                                                                                                                 │
│ Not yet separate agents                                                                                         │
│                                                                                                                 │
│ IntakeAgent                                                                                                     │
│                                                                                                                 │
│ SourcePlanningAgent                                                                                             │
│                                                                                                                 │
│ Not yet fully implemented capabilities                                                                          │
│                                                                                                                 │
│ explicit news                                                                                                   │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│ style article generation                                                                                        │
│                                                                                                                 │
│ comparison mode                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Chunk 18 ────────────────────────────────────────────────────╮
│ true LLM                                                                                                        │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│ based ranking adjustment                                                                                        │
│                                                                                                                 │
│ true LLM-based synthesis backed by external model provider                                                      │
│                                                                                                                 │
│ persistent database storage                                                                                     │
│                                                                                                                 │
│ authentication / multi                                                                                          │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│ user support                                                                                                    │
│                                                                                                                 │
│ richer review metrics                                                                                           │
│                                                                                                                 │
│ artifact download endpoints in FastAPI                                                                          │
│                                                                                                                 │
│ Required Rules for Future Changes                                                                               │
│                                                                                                                 │
│ When updating the codebase, keep AGENTS.md synchronized by following these rules:                               │
│                                                                                                                 │
│ Do not document an agent that does not exist in code.                                                           │
│                                                                                                                 │
│ If a new agent is introduced, add:                                                                              │
│                                                                                                                 │
│ module path                                                                                                     │
│                                                                                                                 │
│ inputs                                                                                                          │
│                                                                                                                 │
│ outputs                                                                                                         │
│                                                       

╭─────────────────────────────────────────────────── Chunk 19 ────────────────────────────────────────────────────╮
│ If an agent is removed or merged, update this file immediately.                                                 │
│                                                                                                                 │
│ If artifact paths change, update this file immediately.                                                         │
│                                                                                                                 │
│ If the orchestrator order changes, update this file immediately.                                                │
│                                                                                                                 │
│ Recommended Next Agent Refactor                                                                                 │
│                                                                                                                 │
│ To better match the product specification, the next clean refactor would be:                                    │
│                                                                                                                 │
│ Extract IntakeAgent from orchestrator.py                                                                        │
│                                                                                                                 │
│ Extract SourcePlanningAgent from orchestrator.py                                                                │
│                                                                                                                 │
│ Rename content.py to derivative_content.py if broader outputs are added                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Chunk 20 ────────────────────────────────────────────────────╮
│ Add ArticleAgent only when article generation actually exists in code                                           │
│                                                                                                                 │
│ Add APIExportAgent only if serving downloadable artifacts becomes separate behavior                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Total Chunks 21


In [41]:
import requests
payload = {
  'tool': 'parse_file',
  'path': '/content/mcp-servers/mcp-unstructured/src/AGENTS_updated.md', #sample_path,
  'route': 'auto',
  'chunking_strategy': 'basic',
  'vlm_mode': USE_VLM,
  'vlm_model_provider': VLM_PROVIDER,
  'vlm_model': VLM_MODEL,
}

resp = requests.post('http://localhost:8000', json=payload, timeout=600 if USE_VLM else 180)
print('status:', resp.status_code)
print(resp.text[:2000])


status: 200
{"result": {"text": "AGENTS.md\n\nPurpose\n\nThis project is an evidence-constrained intelligence platform that retrieves recent signals from multiple sources, ranks and deduplicates them, generates grounded synthesis, creates derivative analyst outputs, exports structured artifacts, and exposes the workflow through a FastAPI backend and Gradio frontend.\n\nThis file is synchronized to the current codebase as implemented in the notebook-generated modules.\n\nCurrent Code\n\n-\n\nAligned Agent Topology\n\nThe code currently implements the following agents under:\n\nbackend/src/intelligence/agents/retrieval.py\n\nbackend/src/intelligence/agents/ranking.py\n\nbackend/src/intelligence/agents/synthesis.py\n\nbackend/src/intelligence/agents/content.py\n\nbackend/src/intelligence/agents/export.py\n\nbackend/src/intelligence/agents/review.py\n\nbackend/src/intelligence/agents/orchestrator.py\n\nImportant synchronization note\n\nThe earlier specification included IntakeAgent and Sou

In [42]:
from rich.panel import Panel
from rich.console import Console

console = Console()
result = resp.json()["result"]

console.print(Panel(result["text"][:1000], title="Document Text"))

for i, chunk in enumerate(result["chunks"][:]):
    console.print(Panel(chunk["text"], title=f"Chunk {i}"))

print(f"Total Chunks {len(result["chunks"])}")

╭───────────────────────────────────────────────── Document Text ─────────────────────────────────────────────────╮
│ AGENTS.md                                                                                                       │
│                                                                                                                 │
│ Purpose                                                                                                         │
│                                                                                                                 │
│ This project is an evidence-constrained intelligence platform that retrieves recent signals from multiple       │
│ sources, ranks and deduplicates them, generates grounded synthesis, creates derivative analyst outputs, exports │
│ structured artifacts, and exposes the workflow through a FastAPI backend and Gradio frontend.                   │
│                                                                                                                 │
│ This file is synchronized to the current codebase as implemented in the notebook-generated modules.             │
│                                                                                                                 │
│ Current Code                                                                                                    │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│ Aligned Agent Topology                                                                                          │
│                                                                                                                 │
│ The code currently implements the following agents under:                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/retrieval.py                                                                    │
│                                                                                                                 │
│ backend/src/intelligence/agents/ranking.py                                                                      │
│                                                                                                                 │
│ backend/src/intelligence/agents/synthesis.py                                                                    │
│                                                                                                                 │
│ backend/src/intelligence/agents/content.py                                                                      │
│                                                                                                                 │
│ backend/src/intelligence/agents/export.py                                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/review.py                                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/orchestrator.py                                                                 │
│                                                                                                                 │
│ Important synchronization note                                                                                  │
│                                                       

╭──────────────────────────────────────────────────── Chunk 0 ────────────────────────────────────────────────────╮
│ AGENTS.md                                                                                                       │
│                                                                                                                 │
│ Purpose                                                                                                         │
│                                                                                                                 │
│ This project is an evidence-constrained intelligence platform that retrieves recent signals from multiple       │
│ sources, ranks and deduplicates them, generates grounded synthesis, creates derivative analyst outputs, exports │
│ structured artifacts, and exposes the workflow through a FastAPI backend and Gradio frontend.                   │
│                                                                                                                 │
│ This file is synchronized to the current codebase as implemented in the notebook-generated modules.             │
│                                                                                                                 │
│ Current Code                                                                                                    │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│ Aligned Agent Topology                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Chunk 1 ────────────────────────────────────────────────────╮
│ The code currently implements the following agents under:                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/retrieval.py                                                                    │
│                                                                                                                 │
│ backend/src/intelligence/agents/ranking.py                                                                      │
│                                                                                                                 │
│ backend/src/intelligence/agents/synthesis.py                                                                    │
│                                                                                                                 │
│ backend/src/intelligence/agents/content.py                                                                      │
│                                                                                                                 │
│ backend/src/intelligence/agents/export.py                                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/review.py                                                                       │
│                                                                                                                 │
│ backend/src/intelligence/agents/orchestrator.py                                                                 │
│                                                                                                                 │
│ Important synchronization note                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Chunk 2 ────────────────────────────────────────────────────╮
│ The earlier specification included IntakeAgent and SourcePlanningAgent. Those are not currently implemented as  │
│ standalone agents in code.                                                                                      │
│                                                                                                                 │
│ Their responsibilities are currently handled inside orchestrator.py by: - constructing ResearchRequest -        │
│ assigning defaults for enabled sources - passing request parameters directly into retrieval                     │
│                                                                                                                 │
│ To stay synchronized with the codebase, this AGENTS file documents the agents that actually exist today.        │
│                                                                                                                 │
│ Agent Contracts                                                                                                 │
│                                                                                                                 │
│ 1. RetrievalAgent                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── Chunk 3 ────────────────────────────────────────────────────╮
│ Module: backend/src/intelligence/agents/retrieval.py                                                            │
│                                                                                                                 │
│ Purpose: Execute multi-source retrieval and persist raw evidence artifacts.                                     │
│                                                                                                                 │
│ Inputs:                                                                                                         │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│  run_dir: str                                                                                                   │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│  query: str                                                                                                     │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│  days: int = 30                                                                                                 │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│  enabled_sources: list | None                                                                                   │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│  verbose: bool = False                                                                                          │
│                                                                                                                 │
│ Primary responsibilities: - call retrieval_service.retrieve(...) - execute enabled source adapters - collect    │
│ raw source items by source - persist raw retrieval output - persist retrieval summary counts                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────── Chunk 17 ────────────────────────────────────────────────────╮
│ A run is considered complete only when: - processed/ranked_items.json exists - synthesis/package.json exists -  │
│ outputs/storyboard.md exists - export/export.json exists - review/review_report.json exists                     │
│                                                                                                                 │
│ Synchronization Gaps Between Spec and Code                                                                      │
│                                                                                                                 │
│ The following earlier-spec items are not yet fully implemented as standalone code components:                   │
│                                                                                                                 │
│ Not yet separate agents                                                                                         │
│                                                                                                                 │
│ IntakeAgent                                                                                                     │
│                                                                                                                 │
│ SourcePlanningAgent                                                                                             │
│                                                                                                                 │
│ Not yet fully implemented capabilities                                                                          │
│                                                                                                                 │
│ explicit news                                                                                                   │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│ style article generation                                                                                        │
│                                                                                                                 │
│ comparison mode                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Chunk 18 ────────────────────────────────────────────────────╮
│ true LLM                                                                                                        │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│ based ranking adjustment                                                                                        │
│                                                                                                                 │
│ true LLM-based synthesis backed by external model provider                                                      │
│                                                                                                                 │
│ persistent database storage                                                                                     │
│                                                                                                                 │
│ authentication / multi                                                                                          │
│                                                                                                                 │
│ -                                                                                                               │
│                                                                                                                 │
│ user support                                                                                                    │
│                                                                                                                 │
│ richer review metrics                                                                                           │
│                                                                                                                 │
│ artifact download endpoints in FastAPI                                                                          │
│                                                                                                                 │
│ Required Rules for Future Changes                                                                               │
│                                                                                                                 │
│ When updating the codebase, keep AGENTS.md synchronized by following these rules:                               │
│                                                                                                                 │
│ Do not document an agent that does not exist in code.                                                           │
│                                                                                                                 │
│ If a new agent is introduced, add:                                                                              │
│                                                                                                                 │
│ module path                                                                                                     │
│                                                                                                                 │
│ inputs                                                                                                          │
│                                                                                                                 │
│ outputs                                                                                                         │
│                                                       

╭─────────────────────────────────────────────────── Chunk 19 ────────────────────────────────────────────────────╮
│ If an agent is removed or merged, update this file immediately.                                                 │
│                                                                                                                 │
│ If artifact paths change, update this file immediately.                                                         │
│                                                                                                                 │
│ If the orchestrator order changes, update this file immediately.                                                │
│                                                                                                                 │
│ Recommended Next Agent Refactor                                                                                 │
│                                                                                                                 │
│ To better match the product specification, the next clean refactor would be:                                    │
│                                                                                                                 │
│ Extract IntakeAgent from orchestrator.py                                                                        │
│                                                                                                                 │
│ Extract SourcePlanningAgent from orchestrator.py                                                                │
│                                                                                                                 │
│ Rename content.py to derivative_content.py if broader outputs are added                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Chunk 20 ────────────────────────────────────────────────────╮
│ Add ArticleAgent only when article generation actually exists in code                                           │
│                                                                                                                 │
│ Add APIExportAgent only if serving downloadable artifacts becomes separate behavior                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Total Chunks 21
